# Проект: Physics-Informed Neural Network для литий-ионной батареи NASA PCoE

Реальный проект модуля 5 на настоящих экспериментальных данных. Работаем с публичным датасетом NASA Ames Prognostics Center of Excellence (PCoE): литиевая батарея 18650 (B0005), которая циклически заряжалась и разряжалась до конца ресурса — 168 разрядов, пока ёмкость не упала с 1.86 до 1.32 Ач.

Проект состоит из двух частей, и обе — обратные задачи, главная сила PINN:

1. **Идентификация физической модели разряда.** По кривым напряжения V(t) строим сеть, которая подчиняется уравнению баланса заряда, и восстанавливаем физические параметры батареи: кривую напряжения разомкнутой цепи OCV(s) и внутреннее сопротивление R0, растущее со старением.
2. **Прогноз остаточного ресурса (RUL).** Обучаем сеть на первых 80 циклах жизни батареи и экстраполируем закон деградации до конца ресурса: предсказываем, на каком цикле ёмкость упадёт ниже порога 70%. Сравниваем с обычными регрессиями без физики.

Такой проект можно показать в портфолио: реальные данные, физическая модель, честная валидация на отложенных циклах и сравнение с бейзлайнами.

**Источники данных:**
- оригинал: https://data.nasa.gov/dataset/li-ion-battery-aging-datasets
- зеркало на GitHub (используется для автозагрузки): https://github.com/changyeon99/Battery-Data-Set
- зеркало на Kaggle: https://www.kaggle.com/datasets/patrickfleith/nasa-battery-dataset

В ноутбуке используется чистый PyTorch — как в модуле 4, без готовых PINN-библиотек, чтобы вся механика была видна.

## 1. Загрузка и разбор данных

Файл B0005.mat — это MATLAB-контейнер с записью всех 616 циклов: заряды, разряды и измерения импеданса. Нас интересуют разряды: в каждом записаны время, напряжение, ток, температура (измеряются датчиками с шумом) и ёмкость, отданная за разряд.

Если файла нет рядом с ноутбуком — он скачивается из зеркала на GitHub (16 МБ).

In [ ]:
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство:', device)

DATA_URL = 'https://raw.githubusercontent.com/changyeon99/Battery-Data-Set/master/B0005.mat'
DATA_PATH = Path('data') / 'B0005.mat'
DATA_PATH.parent.mkdir(exist_ok=True)
if not DATA_PATH.exists():
    print('Скачиваем датасет NASA PCoE...')
    with urlopen(DATA_URL) as r, open(DATA_PATH, 'wb') as f:
        f.write(r.read())
print('Файл данных:', DATA_PATH.resolve())

In [ ]:
mat = sio.loadmat(DATA_PATH)
cycles = mat['B0005'][0, 0]['cycle']

discharges = []
for i in range(cycles.shape[1]):
    if str(cycles[0, i]['type'][0]) != 'discharge':
        continue
    d = cycles[0, i]['data'][0, 0]
    t = d['Time'].flatten().astype(float)
    if len(t) < 100:  # пропускаем вырожденные записи
        continue
    discharges.append({
        't': t,
        'V': d['Voltage_measured'].flatten().astype(float),
        'I': float(np.abs(d['Current_measured'].flatten().astype(float).mean())),
        'Q': float(d['Capacity'].flatten()[0]),
    })

N = len(discharges)          # число валидных разрядов
Z_MAX = float(N)
z = np.arange(1, N + 1)      # номера циклов разряда
Q_arr = np.array([d['Q'] for d in discharges])
I_arr = np.array([d['I'] for d in discharges])
TE_arr = np.array([d['t'].max() for d in discharges])
Q_FAIL = 1.4                 # порог отказа: 70% от 2 Ач
rul_true = int(z[np.argmax(Q_arr < Q_FAIL)])
print(f'Валидных разрядов: {N}')
print(f'Ёмкость: {Q_arr[0]:.4f} -> {Q_arr[-1]:.4f} Ач')
print(f'Ток разряда: {I_arr.mean():.3f} А (постоянный)')
print(f'Истинный RUL (цикл, где Q < {Q_FAIL} Ач): {rul_true}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(z, Q_arr, 'C0.', markersize=3)
axes[0].axhline(Q_FAIL, color='r', ls='--', label=f'порог отказа {Q_FAIL} Ач')
axes[0].set_xlabel('цикл разряда')
axes[0].set_ylabel('ёмкость, Ач')
axes[0].set_title('Деградация ёмкости батареи B0005')
axes[0].legend()
axes[0].grid(alpha=0.3)
for k, color in zip([1, 50, 100, 150], ['C0', 'C1', 'C2', 'C3']):
    d = discharges[k]
    axes[1].plot(d['t'] / 60, d['V'], color=color, label=f'цикл {k + 1}')
axes[1].set_xlabel('время разряда, мин')
axes[1].set_ylabel('напряжение, В')
axes[1].set_title('Кривые разряда (настоящие измерения)')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.show()

На левом графике — главная драма литий-ионных батарей: ёмкость монотонно падает с ускорением. На правом — кривые разряда при постоянном токе ~2 А: длинное пологое плато и резкий спад в конце (колено). С ростом номера цикла плато проседает — это рост внутренних потерь.

Именно эти два наблюдения мы и опишем физической моделью.

## 2. Часть A. Физическая модель разряда: постановка

Эквивалентная схема батареи при разряде постоянным током I:

V(t) = OCV(s(t)) - I * R0

где s(t) — степень заряженности (SoC, от 1 до 0), OCV(s) — напряжение разомкнутой цепи (физическая характеристика химии батареи), R0 — омическое внутреннее сопротивление. Баланс заряда даёт дифференциальное уравнение:

ds/dt = -I / (3600 * Q),  где Q — ёмкость в Ач, I — в А, t — в секундах.

Что неизвестно: кривая OCV(s) и зависимость R0 от возраста батареи. Их мы и восстанавливаем из данных — это типичная обратная задача.

Конструкция PINN:
- сеть s(tau, zeta) предсказывает SoC по нормированному времени разряда tau = t/t_end и нормированному возрасту zeta = cycle/N;
- физический loss: residual уравнения баланса заряда в случайных точках (collocation points), производная ds/dt берётся через torch.autograd.grad — так же, как в примерах с уравнением теплопроводности;
- начальное условие: s(0, zeta) = 1;
- данные: предсказанное напряжение V = OCV(s) - I*R0 должно совпадать с измеренным.

OCV(s) — монотонно растущая кривая: задаём 60 узлов по s (гуще на концах, где кривая круче) и обучаем приращения через softplus — монотонность соблюдена по построению. R0(zeta) — кусочно-линейная монотонная функция из 8 узлов: сопротивление может только расти со старением.

ВАЖНО! В честной валидации участвуют только 4 из 5 циклов; каждый пятый цикл отложен (holdout) — модель его не видит при обучении, и мы проверяем предсказание кривой напряжения на нём.

In [ ]:
# Сборка тензоров: одна точка = (tau, zeta, V, номер цикла j)
taus, zetas, Vs, js = [], [], [], []
for j, d in enumerate(discharges):
    taus.append(d['t'] / d['t'].max())
    zetas.append(np.full(len(d['t']), (j + 1) / Z_MAX))
    Vs.append(d['V'])
    js.append(np.full(len(d['t']), j))
tau_d = torch.tensor(np.concatenate(taus), dtype=torch.float32).view(-1, 1)
zeta_d = torch.tensor(np.concatenate(zetas), dtype=torch.float32).view(-1, 1)
V_d = torch.tensor(np.concatenate(Vs), dtype=torch.float32).view(-1, 1)
j_d = torch.tensor(np.concatenate(js), dtype=torch.long)
I_t = torch.tensor(I_arr)
Q_t = torch.tensor(Q_arr)
TE_t = torch.tensor(TE_arr)

holdout = torch.tensor(sorted(range(4, N, 5)), dtype=torch.long)  # каждый пятый цикл
in_train = ~torch.isin(j_d, holdout)
tau_tr, zeta_tr, V_tr, j_tr = tau_d[in_train], zeta_d[in_train], V_d[in_train], j_d[in_train]
n_pts = len(V_tr)
print(f'Точек данных: всего {len(V_d)}, на обучении {n_pts}, на holdout {len(V_d) - n_pts}')

In [ ]:
H = 64


class SoCNet(nn.Module):
    """Сеть SoC: вход (tau, zeta) -> s."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, H), nn.Tanh(),
            nn.Linear(H, H), nn.Tanh(),
            nn.Linear(H, H), nn.Tanh(),
            nn.Linear(H, 1),
        )
        nn.init.zeros_(self.net[-1].weight)  # стартуем с s = 0, физика быстро выпрямит
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, tau, zeta):
        return self.net(torch.cat([tau, zeta], dim=1))


K = 60
# узлы OCV: косинусная сетка — гуще у s = 0 и s = 1, где кривая крутая
s_grid_t = torch.tensor(((1 - np.cos(np.pi * np.arange(K) / (K - 1))) / 2).astype(np.float32))


def make_ocv(c0, delta):
    """Монотонная кусочно-линейная OCV: приращения = softplus(delta)."""

    def ocv(s):
        inc = torch.nn.functional.softplus(delta)
        vals = c0 + torch.cumsum(inc, 0) - inc[0]
        idx = torch.clamp(torch.searchsorted(s_grid_t[1:], s.view(-1)).long(), 0, K - 2)
        s0, s1 = s_grid_t[idx], s_grid_t[idx + 1]
        frac = ((s.view(-1) - s0) / (s1 - s0)).clamp(0, 1)
        return (vals[idx] * (1 - frac) + vals[idx + 1] * frac).view(-1, 1)

    return ocv


M = 8
z_grid = torch.linspace(0, 1, M)


def make_r0(ra, rb):
    """Монотонно растущее с возрастом сопротивление R0(zeta)."""

    def r0(zeta):
        inc = torch.nn.functional.softplus(rb)
        vals = torch.nn.functional.softplus(ra) + torch.cumsum(inc, 0) - inc[0]
        pos = zeta.view(-1) * (M - 1)
        idx = torch.clamp(pos.long(), 0, M - 2)
        frac = pos - idx.float()
        return (vals[idx] * (1 - frac) + vals[idx + 1] * frac).view(-1, 1)

    return r0


soc_net = SoCNet()
V_MIN, V_MAX = 2.7, 4.19
c0 = nn.Parameter(torch.tensor(float(V_MIN)))
delta = nn.Parameter(torch.full((K,), float(np.log(np.expm1((V_MAX - V_MIN) / (K - 1))))))
ra = nn.Parameter(torch.tensor(np.log(0.03)))
rb = nn.Parameter(torch.full((M,), float(np.log(np.expm1(0.006)))))
ocv = make_ocv(c0, delta)
r0 = make_r0(ra, rb)
print('Параметров в сети SoC:', sum(p.numel() for p in soc_net.parameters()))

### Обучение

Обучение — minibatch, как принято в глубоком обучении: на каждом шаге берём случайный батч точек данных и свежий набор collocation-точек для физического residual. Всего 10 000 шагов Adam; на CPU это занимает 5–7 минут.

loss = w_f * physics + w_ic * IC + data

In [ ]:
params = list(soc_net.parameters()) + [c0, delta, ra, rb]
optimizer = torch.optim.Adam(params, lr=1e-3)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[6000, 8500], gamma=0.3)

STEPS, B_DATA, B_F = 10000, 4096, 1024
history = []
for step in range(1, STEPS + 1):
    optimizer.zero_grad()
    # данные: случайный батч измерений
    sel = torch.randint(0, n_pts, (B_DATA,))
    tau_b, zeta_b, V_b, j_b = tau_tr[sel], zeta_tr[sel], V_tr[sel], j_tr[sel]
    # физика: случайные циклы и моменты времени
    jf = torch.randint(0, N, (B_F,))
    tau_f = torch.rand(B_F, 1)
    tau_f.requires_grad_(True)
    zeta_f = (jf.float() + 1).view(-1, 1) / N
    s_f = soc_net(tau_f, zeta_f)
    ds_dtau = torch.autograd.grad(s_f, tau_f, torch.ones_like(s_f), create_graph=True)[0]
    ds_dt = ds_dtau / TE_t[jf].view(-1, 1)  # переход от d/d(tau) к d/dt
    rhs = I_t[jf].view(-1, 1) / (3600 * Q_t[jf].view(-1, 1))
    loss_f = torch.mean((ds_dt + rhs) ** 2)
    # начальное условие s(0, zeta) = 1
    jic = torch.randint(0, N, (256,))
    loss_ic = torch.mean((soc_net(torch.zeros(256, 1), (jic.float() + 1).view(-1, 1) / N) - 1) ** 2)
    # данные: напряжение
    V_pred = ocv(soc_net(tau_b, zeta_b)) - I_t[j_b].view(-1, 1) * r0(zeta_b)
    loss_data = torch.mean((V_pred - V_b) ** 2)
    loss = 10 * loss_f + 10 * loss_ic + loss_data
    loss.backward()
    optimizer.step()
    scheduler.step()
    history.append(loss_data.item())
    if step % 1000 == 0:
        print(f'шаг {step:6d} | data = {loss_data.item():.3e} | physics = {loss_f.item():.3e}')

### Результаты части A

In [ ]:
with torch.no_grad():
    s_all = soc_net(tau_d, zeta_d)
    V_hat = ocv(s_all) - I_t[j_d].view(-1, 1) * r0(zeta_d)
    m_ho = torch.isin(j_d, holdout)
    rmse_ho = torch.sqrt(torch.mean((V_hat[m_ho] - V_d[m_ho]) ** 2)).item()
    rmse_tr = torch.sqrt(torch.mean((V_hat[~m_ho] - V_d[~m_ho]) ** 2)).item()
print(f'RMSE напряжения: обучение {rmse_tr * 1000:.1f} мВ, holdout {rmse_ho * 1000:.1f} мВ')
print(f'Идентифицированное R0: {r0(torch.zeros(1, 1)).item():.4f} Ом в начале жизни -> '
      f'{r0(torch.ones(1, 1)).item():.4f} Ом в конце')
for lo, hi in [(0, 40), (40, 80), (80, 120), (120, 168)]:
    m = (j_d >= lo) & (j_d < hi)
    r = torch.sqrt(torch.mean((V_hat[m] - V_d[m]) ** 2)).item()
    print(f'  циклы {lo + 1:3d}-{hi:3d}: RMSE = {r * 1000:.1f} мВ')

In [ ]:
# Кривые разряда: измерения против модели (включая holdout-циклы)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
for ax, k in zip(axes, [4, 44, 99, 149]):
    d = discharges[k]
    tt = torch.tensor(d['t'] / d['t'].max(), dtype=torch.float32).view(-1, 1)
    zz = torch.full_like(tt, (k + 1) / Z_MAX)
    with torch.no_grad():
        v_model = (ocv(soc_net(tt, zz)) - d['I'] * r0(zz)).flatten()
    tag = ' (holdout)' if k in holdout.tolist() else ''
    ax.plot(d['t'] / 60, d['V'], 'C0.', markersize=2, label='измерения')
    ax.plot(d['t'] / 60, v_model, 'r-', lw=2, label='PINN')
    ax.set_title(f'цикл {k + 1}{tag}')
    ax.set_xlabel('время, мин')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('напряжение, В')
axes[0].legend()
plt.show()

In [ ]:
# Восстановленные физические характеристики батареи
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
s_line = torch.linspace(0, 1, 301).view(-1, 1)
with torch.no_grad():
    axes[0].plot(s_line, ocv(s_line), 'C0')
axes[0].set_xlabel('SoC')
axes[0].set_ylabel('OCV, В')
axes[0].set_title('Восстановленная кривая OCV(s)')
axes[0].grid(alpha=0.3)
z_line = torch.linspace(0, 1, 201).view(-1, 1)
with torch.no_grad():
    axes[1].plot(z_line * N + 1, r0(z_line), 'C1')
axes[1].set_xlabel('цикл разряда')
axes[1].set_ylabel('R0, Ом')
axes[1].set_title('Восстановленный рост внутреннего сопротивления')
axes[1].grid(alpha=0.3)
plt.show()

Что получилось и что это значит:

- Ошибка модели — десятки милливольт на всём сроке службы, при том что holdout-циклы модель не видела. Собственный шум датчика напряжения в этом датасете — единицы-десятки мВ, так что модель вышла на уровень, где дальше упереться можно только в шум и в упрощённость схемы (температуру мы не учитываем, а батарея за разряд греется до 35-38 °C).
- Идентифицированная OCV(s) имеет каноническую для Li-ion форму: круче на концах, пологое плато в середине.
- R0 растёт с возрастом примерно вдвое-втрое за срок службы — это известный физический механизм деградации (рост пассивационных плёнок и потеря контакта активного материала).

Это и есть решение обратной задачи: из «чёрного ящика» с датчиками извлечены физически осмысленные характеристики.

## 3. Часть B. Прогноз остаточного ресурса (RUL)

Теперь задача прогноза: обучаемся только на первых 80 циклах (меньше половины жизни батареи) и предсказываем, когда ёмкость пересечёт порог отказа 1.4 Ач. Истинный ответ (его модель не видит): цикл 125.

Физическое ядро — эмпирический закон деградации: скорость потери ёмкости растёт со старением (деградация SEI-плёнки плюс потеря активного материала):

dQ/dz = -(k1 + 2 * k2 * z),  k1, k2 > 0

Сеть Q(zeta) приближает зависимость ёмкости от номера цикла, физический loss требует выполнения закона, данные — совпадения с измерениями на первых 80 циклах. Коэффициенты k1, k2 обучаются вместе с сетью и имеют физический смысл (линейная и ускоряющаяся части деградации).

ВАЖНО! Обучаемые k1, k2 — маленькие числа (порядка 0.003 и 0.00001). Если обучать их напрямую, градиенты затухают. Стандартный приём — параметризация через логарифм: k = exp(theta), тогда theta может гулять свободно, а k остаётся положительным.

Бейзлайны для сравнения — регрессии без физики, обученные на тех же 80 циклах: линейная и полином 3-й степени.

In [ ]:
N_TRAIN = 80


class FadeNet(nn.Module):
    """Ёмкость как функция нормированного возраста zeta = z / N."""

    def __init__(self, q0):
        super().__init__()
        self.q0 = nn.Parameter(torch.tensor(float(q0)))
        self.net = nn.Sequential(
            nn.Linear(1, 32), nn.Tanh(),
            nn.Linear(32, 32), nn.Tanh(),
            nn.Linear(32, 1),
        )
        nn.init.zeros_(self.net[-1].weight)  # стартуем с константы Q0
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, zeta):
        return self.q0 + self.net(zeta)


fade_net = FadeNet(Q_arr[0])
theta1 = nn.Parameter(torch.tensor(np.log(3e-3)))
theta2 = nn.Parameter(torch.tensor(np.log(1e-5)))
opt_b = torch.optim.Adam(list(fade_net.parameters()) + [theta1, theta2], lr=1e-3)

zeta_train = torch.tensor(z[:N_TRAIN] / Z_MAX, dtype=torch.float32).view(-1, 1)
Q_train = torch.tensor(Q_arr[:N_TRAIN], dtype=torch.float32).view(-1, 1)

for ep in range(1, 8001):
    opt_b.zero_grad()
    # физика: collocation в области обучения [0, N_TRAIN/N]
    zf = torch.rand(400, 1) * (N_TRAIN / Z_MAX)
    zf.requires_grad_(True)
    dQ = torch.autograd.grad(fade_net(zf), zf, torch.ones_like(zf), create_graph=True)[0]
    k1, k2 = theta1.exp(), theta2.exp()
    # dQ/dzeta = dQ/dz * dz/dzeta = (k1 + 2 k2 z) * N
    rhs = (k1 + 2 * k2 * zf * Z_MAX) * Z_MAX
    loss_f = torch.mean((dQ + rhs) ** 2)
    loss_ic = (fade_net(torch.zeros(1, 1)) - Q_arr[0]) ** 2
    loss_data = torch.mean((fade_net(zeta_train) - Q_train) ** 2)
    loss = 10 * loss_f + 10 * loss_ic + loss_data
    loss.backward()
    opt_b.step()
    if ep % 2000 == 0:
        print(f'эпоха {ep:5d} | data = {loss_data.item():.3e} | physics = {loss_f.item():.3e}')

print(f'\nИдентифицированные коэффициенты: k1 = {theta1.exp().item():.6f} Ач/цикл, '
      f'k2 = {theta2.exp().item():.3e} Ач/цикл^2')

In [ ]:
z_line = torch.tensor(z / Z_MAX, dtype=torch.float32).view(-1, 1)
with torch.no_grad():
    Q_pinn = fade_net(z_line).flatten().numpy()

# бейзлайны без физики: линейная регрессия и полином 3-й степени на тех же 80 циклах
coef_lin = np.polyfit(z[:N_TRAIN], Q_arr[:N_TRAIN], 1)
coef_poly = np.polyfit(z[:N_TRAIN], Q_arr[:N_TRAIN], 3)
Q_lin = np.polyval(coef_lin, z)
Q_poly = np.polyval(coef_poly, z)


def rul_of(q_pred):
    hit = np.argmax(q_pred < Q_FAIL) if (q_pred < Q_FAIL).any() else -1
    return int(z[hit]) if hit >= 0 else None

print(f'Истинный RUL: цикл {rul_true}')
print(f'PINN:        цикл {rul_of(Q_pinn)} (ошибка {abs(rul_of(Q_pinn) - rul_true)} цикл.)')
print(f'линейная:    цикл {rul_of(Q_lin)} (ошибка {abs(rul_of(Q_lin) - rul_true)} цикл.)')
print(f'полином 3:   цикл {rul_of(Q_poly)} (ошибка {abs(rul_of(Q_poly) - rul_true)} цикл.)')
print(f'MAE ёмкости на всех 168 циклах: PINN {np.abs(Q_pinn - Q_arr).mean() * 1000:.1f} мАч, '
      f'линейная {np.abs(Q_lin - Q_arr).mean() * 1000:.1f} мАч, '
      f'полином 3 {np.abs(Q_poly - Q_arr).mean() * 1000:.1f} мАч')

In [ ]:
plt.figure(figsize=(9, 4.2))
plt.plot(z, Q_arr, 'k.', markersize=4, label='измерения')
plt.plot(z, Q_pinn, 'C0', lw=2, label='PINN (обучение на первых 80 циклах)')
plt.plot(z, Q_lin, 'C2--', label='линейная регрессия')
plt.plot(z, Q_poly, 'C3--', label='полином 3-й степени')
plt.axvline(N_TRAIN, color='gray', ls=':', label='граница обучения')
plt.axhline(Q_FAIL, color='r', ls='--', label=f'порог отказа {Q_FAIL} Ач')
plt.xlabel('цикл разряда')
plt.ylabel('ёмкость, Ач')
plt.title('Прогноз деградации и остаточного ресурса (RUL)')
plt.legend(loc='lower left')
plt.grid(alpha=0.3)
plt.show()

Почему это интересно:

- Линейная регрессия не знает про ускорение деградации и систематически опаздывает с прогнозом отказа. Полином без физики, наоборот, слишком «уверен» в экстраполяции и уходит от данных — старая история про полиномы за пределами обучающей выборки.
- PINN удерживает форму закона и за границей обучения: физический residual не даёт сети выдумывать поведение там, где данных нет. Ошибка прогноза RUL — единицы циклов на горизонте в полсотни.
- Вместе с частью A получается связная история: модель разряда объясняет, *почему* проседает напряжение (рост R0), а модель деградации — *когда* батарея откажет.

## 4. Итоги и что развить самостоятельно

В проекте пройден полный цикл работы с физически информированными сетями на реальных данных:

1. Разбор реального формата данных (MATLAB-контейнер, зашумлённые датчики, выбросы-записи).
2. Формулировка прямых и обратных задач: восстановление OCV и R0 из кривых разряда, восстановление коэффициентов закона деградации.
3. Реализация PINN на чистом PyTorch: autograd для residual, collocation-точки, взвешивание слагаемых loss, лог-параметризация малых коэффициентов.
4. Честная валидация: holdout-циклы, сравнение с бейзлайнами, экстраполяция за границу обучения.

Направления для развития (любое из них усиливает портфолио):

- добавить температуру третьим входом сети s(tau, zeta, T) и проверить, снизится ли ошибка;
- повторить анализ на батареях B0006 и B0007 из того же датасета и сравнить скорость деградации;
- заменить закон деградации на двухэкспоненциальный (как в статьях по RUL) и сравнить точность;
- обучить один PINN сразу на трёх батареях и оценить переносимость между ячейками;
- сравнить с классическим ML из модулей 2-3: XGBoost по признакам цикла против PINN;
- добавить оценку неопределённости: ансамбль из нескольких PINN с разными seed, доверительный интервал прогноза RUL.